## Imports

In [ ]:
import asyncio, json, os, re, secrets, uuid
from pathlib import Path
from urllib.parse import quote

import dialoghelper as dh
from fasthtml.common import *
from fasthtml.jupyter import *

## Restart FastHTML Server

In [ ]:
import subprocess

port = 8000
def kill_port(port=port):
    subprocess.run(f"lsof -ti:{port} | xargs -r kill -9", shell=True)

kill_port()

## 1.0 FastHTML app and shared route helpers

In [ ]:
if not globals().get('srv'):
    app = FastHTML(session_cookie='solveit_social_session')
    rt = app.route
    srv = JupyUvi(app)
#elif 'rt' not in globals():
#    rt = app.route

SOCIAL_PORT = getattr(srv, 'port', 8000)
_domains = json.loads(os.environ.get('PUBLIC_DOMAINS', '{}'))
public_domain = _domains.get(str(SOCIAL_PORT), globals().get('public_domain', ''))
if not public_domain:
    raise RuntimeError(f'No public domain is configured for port {SOCIAL_PORT}.')

def _drop_route(app, path, method=None):
    if app is not None:
        app.routes[:] = [route for route in app.routes if not (
            getattr(route, 'path', None) == path and
            (method is None or method in (getattr(route, 'methods', set()) or set())))]

def _endpoint(domain, path):
    base = str(domain or '').rstrip('/')
    if not base: return path
    return (base if base.startswith(('http://', 'https://')) else f'https://{base}') + path

def _cors(req):
    return {'Access-Control-Allow-Origin': req.headers.get('origin') or '*',
            'Vary': 'Origin', 'Cache-Control': 'no-store'}

## 2.0 Read-only dialog-folder media browser

In [ ]:
IMAGES = {'.gif', '.jpeg', '.jpg', '.png', '.webp'}
VIDEOS = {'.m4v', '.mov', '.mp4', '.webm'}
MEDIA_ROUTE = '/social-media/files'

def install_social_files(rt, public_domain=None, path=MEDIA_ROUTE, app=None):
    app = app or getattr(rt, '__self__', None)
    _drop_route(app, path, 'GET')

    @rt(path)
    async def get(req, path: str = ''):
        root = Path(await dh.realpath(Path(dh.find_dname()).parent.as_posix())).resolve()
        data_root = Path(await dh.realpath('/')).resolve()
        relative, folder = Path(path or '.'), None
        try: folder = (root / relative).resolve()
        except OSError: pass
        if (relative.is_absolute() or folder is None or
            not folder.is_relative_to(root) or not folder.is_dir()):
            return JSONResponse({'error': 'Invalid folder.'}, status_code=400, headers=_cors(req))

        items = []
        for item in folder.iterdir():
            try: resolved = item.resolve()
            except OSError: continue
            if item.name.startswith('.') or not resolved.is_relative_to(root): continue
            suffix = item.suffix.lower()
            kind = ('folder' if item.is_dir() else 'gif' if suffix == '.gif' else
                    'image' if suffix in IMAGES else 'video' if suffix in VIDEOS else '')
            if not kind: continue
            entry = {'name': item.name, 'path': item.relative_to(root).as_posix(), 'kind': kind}
            if kind != 'folder':
                if not resolved.is_relative_to(data_root): continue
                entry['url'] = '/static/' + quote(resolved.relative_to(data_root).as_posix(), safe='/')
            items.append(entry)

        current = folder.relative_to(root).as_posix()
        current = '' if current == '.' else current
        parent = None if not current else Path(current).parent.as_posix()
        if parent == '.': parent = ''
        items.sort(key=lambda item: (item['kind'] != 'folder', item['name'].lower()))
        return JSONResponse({'path': current, 'parent': parent, 'items': items}, headers=_cors(req))

    iife(f'window.SOLVEIT_MEDIA_API_URL={json.dumps(_endpoint(public_domain, path))}')

install_social_files(rt, public_domain, app=app)

## 3.0 Twitter payload validation, media inspection, and API helpers

In [ ]:
PUBLISH_ROUTE = '/social-media/publish'
SOLVEIT_SOCIAL_PUBLISH_TOKEN = globals().get('SOLVEIT_SOCIAL_PUBLISH_TOKEN') or secrets.token_urlsafe(32)
SOLVEIT_SOCIAL_PUBLISH_LOCK = globals().get('SOLVEIT_SOCIAL_PUBLISH_LOCK') or asyncio.Lock()
SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS = globals().get('SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS') or {}
MB = 1024 * 1024
LIVE_MEDIA = {
    'image/jpeg': ('image', '.jpg', 5 * MB), 'image/png': ('image', '.png', 5 * MB),
    'image/webp': ('image', '.webp', 5 * MB), 'image/gif': ('gif', '.gif', 15 * MB),
    'video/mp4': ('video', '.mp4', 512 * MB), 'video/quicktime': ('video', '.mov', 512 * MB),
    'video/x-m4v': ('video', '.m4v', 512 * MB), 'video/webm': ('video', '.webm', 512 * MB),
}
MAX_LIVE_MEDIA = max(value[2] for value in LIVE_MEDIA.values())

def _live_plan(body, media_count=0):
    """Revalidate a confirmed live thread before any X request."""
    if not isinstance(body, dict): raise ValueError('A JSON object is required.')
    thread, validation, confirmation = body.get('thread'), body.get('validation'), body.get('confirmation')
    if not isinstance(thread, dict) or not isinstance(validation, dict):
        raise ValueError('thread and validation objects are required.')
    posts, metrics = thread.get('posts'), validation.get('posts')
    if (thread.get('schemaVersion') != 1 or thread.get('platform') != 'x' or
        thread.get('charLimit') != 280 or not isinstance(posts, list) or not 1 <= len(posts) <= 100 or
        not isinstance(metrics, list) or len(metrics) != len(posts)):
        raise ValueError('Unsupported, malformed, or stale thread payload.')
    if not isinstance(confirmation, dict) or confirmation.get('action') != 'publish_to_x' or confirmation.get('postCount') != len(posts):
        raise ValueError('Explicit publishing confirmation is required.')
    expected_type = 'single' if len(posts) == 1 else 'thread'
    if thread.get('postType') != expected_type: raise ValueError(f'postType must be {expected_type}.')

    plan, ids, total = [], set(), 0
    for index, (post, metric) in enumerate(zip(posts, metrics), 1):
        if (not isinstance(post, dict) or not isinstance(post.get('text'), str) or
            not isinstance(post.get('media'), list) or not isinstance(metric, dict)):
            raise ValueError(f'Post {index} is malformed.')
        weighted, remaining = metric.get('weightedLength'), metric.get('remaining')
        if (not isinstance(weighted, int) or isinstance(weighted, bool) or weighted < 0 or
            not isinstance(remaining, int) or remaining != 280 - weighted):
            raise ValueError(f'Post {index} validation is stale.')
        if weighted > 280: raise ValueError(f'Post {index} is over its character limit.')
        client_id, text, items = str(post.get('clientId') or ''), post['text'], post['media']
        if not client_id or client_id in ids: raise ValueError(f'Post {index} needs a unique clientId.')
        if not text.strip() and not items: raise ValueError(f'Post {index} is empty.')
        if len(items) > 4: raise ValueError(f'Post {index} has more than four media items.')
        ids.add(client_id); refs, media = set(), []
        for media_index, item in enumerate(items, 1):
            if not isinstance(item, dict) or item.get('kind') not in ('image', 'gif', 'video'):
                raise ValueError(f'Post {index}, media {media_index} is invalid.')
            ref = str(item.get('ref') or '')
            if not ref or item.get('missing') or ref in refs:
                raise ValueError(f'Post {index} has unresolved or duplicate media.')
            refs.add(ref); media.append({'kind': item['kind'], 'altText': str(item.get('altText') or '')})
        total += len(media); plan.append({'clientId': client_id, 'text': text, 'media': media})
    if media_count != total: raise ValueError('The uploaded media does not match the posting plan.')
    return plan

def _x_auth():
    try: import tweepy
    except ImportError as error: raise RuntimeError('Install Tweepy 4.14 or newer before live publishing.') from error
    names = ('X_API_KEY', 'X_API_SECRET', 'X_ACCESS_TOKEN', 'X_ACCESS_TOKEN_SECRET')
    missing = [name for name in names if not os.environ.get(name)]
    if missing: raise RuntimeError('Missing X credentials: ' + ', '.join(missing))
    return tweepy, [os.environ[name] for name in names]

def _x_client():
    tweepy, values = _x_auth()
    return tweepy.Client(consumer_key=values[0], consumer_secret=values[1],
        access_token=values[2], access_token_secret=values[3], wait_on_rate_limit=False)

def _x_media_api():
    tweepy, values = _x_auth()
    return tweepy.API(tweepy.OAuth1UserHandler(*values))

def _media_kind(head):
    if head.startswith(b'\xff\xd8\xff'): return 'image', 'image/jpeg'
    if head.startswith(b'\x89PNG\r\n\x1a\n'): return 'image', 'image/png'
    if head[:4] == b'RIFF' and head[8:12] == b'WEBP': return 'image', 'image/webp'
    if head.startswith((b'GIF87a', b'GIF89a')): return 'gif', 'image/gif'
    if head[4:8] == b'ftyp': return 'video', 'video/mp4'
    if head.startswith(b'\x1aE\xdf\xa3'): return 'video', 'video/webm'
    return '', ''

async def _read_live_media(upload, planned):
    """Inspect an UploadFile without loading large videos into memory."""
    file = getattr(upload, 'file', None)
    if file is None: raise ValueError('An uploaded media item could not be read.')
    def inspect():
        file.seek(0, 2); size = file.tell(); file.seek(0); head = file.read(16); file.seek(0)
        return size, head
    size, head = await asyncio.to_thread(inspect)
    if not size: raise ValueError('A selected media item is empty.')
    kind, detected = _media_kind(head)
    supplied = str(getattr(upload, 'content_type', '') or '').lower().split(';')[0]
    if not kind: raise ValueError('Use a JPEG, PNG, WebP, GIF, MP4, MOV, M4V, or WebM file.')
    if kind != planned['kind']:
        raise ValueError(f"Media marked as {planned['kind']} does not match its contents.")
    if supplied in LIVE_MEDIA and LIVE_MEDIA[supplied][0] != kind:
        raise ValueError('A selected media type does not match its contents.')
    mime = supplied if supplied in LIVE_MEDIA else detected
    limit = LIVE_MEDIA[mime][2]
    if size > limit: raise ValueError(f'The selected {kind} is larger than {limit // MB} MB.')
    original = re.split(r'[/\\]', str(getattr(upload, 'filename', '') or kind))[-1]
    stem = re.sub(r'[^A-Za-z0-9._-]+', '_', re.sub(r'\.[^.]*$', '', original))[:80] or kind
    return {'file': file, 'kind': kind, 'name': stem + LIVE_MEDIA[mime][1],
            'altText': str(planned.get('altText') or '').strip()}

def _x_post_id(response):
    data = response.get('data', response) if isinstance(response, dict) else getattr(response, 'data', None)
    value = data.get('id') if isinstance(data, dict) else getattr(data, 'id', None)
    if not value: raise RuntimeError('X returned no post ID.')
    return str(value)

def _x_error(error):
    response = getattr(error, 'response', None)
    status, message = getattr(response, 'status_code', None), ''
    try:
        data = response.json()
        message = data.get('detail') or data.get('title') or next((x.get('message', '') for x in data.get('errors', [])), '')
    except Exception: pass
    return status, re.sub(r'\s+', ' ', message or str(error) or 'X rejected the request.').strip()[:500]

def _failure(message, status=500, **details):
    result = {'live': True, 'success': False, 'complete': False, 'safeToRetry': True,
              'resultUnknown': False, 'error': message, 'posts': [], 'sideEffects': [],
              'retryPerformed': False}
    return result | details, status

def _publish_plan(plan, client=None, media_api=None, media=None):
    """Upload all media, then create a non-retried reply chain in order."""
    media = media or [[] for _ in plan]
    if len(media) != len(plan) or any(len(files) != len(step['media']) for files, step in zip(media, plan)):
        return _failure('Prepared media does not match the posting plan.', 422, xAttempted=False)
    if client is None:
        try: client = _x_client()
        except Exception as error:
            _, message = _x_error(error)
            return _failure(message, xAttempted=False)
    effects, media_ids = [], [[] for _ in plan]
    if any(media):
        if media_api is None:
            try: media_api = _x_media_api()
            except Exception as error:
                _, message = _x_error(error)
                return _failure(message, xAttempted=False, postAttempted=False)
        for post_index, files in enumerate(media):
            for media_index, item in enumerate(files):
                try:
                    chunked, category = item['kind'] != 'image', f"tweet_{item['kind']}"
                    uploaded = media_api.media_upload(filename=item['name'], file=item['file'], chunked=chunked,
                        media_category=category, **({'wait_for_async_finalize': True} if chunked else {}))
                    processing = getattr(uploaded, 'processing_info', None) or {}
                    problem = processing.get('error')
                    if problem: raise RuntimeError(problem.get('message') if isinstance(problem, dict) else str(problem))
                    if processing.get('state') not in (None, 'succeeded'): raise RuntimeError('X did not finish processing the media.')
                    media_id = str(getattr(uploaded, 'media_id_string', None) or getattr(uploaded, 'media_id', ''))
                    if not media_id: raise RuntimeError('X returned no media ID.')
                    media_ids[post_index].append(media_id)
                    effects.append({'action': 'upload_media', 'id': media_id, 'postStep': post_index + 1,
                                    'mediaIndex': media_index, 'kind': item['kind']})
                    if item['altText'] and item['kind'] != 'video':
                        media_api.create_media_metadata(media_id, item['altText'][:1000])
                except Exception as error:
                    status, message = _x_error(error)
                    return _failure(message, 502, xAttempted=True, postAttempted=False,
                        mediaUploadAttempted=True, failedStep=post_index + 1, failedMedia=media_index,
                        xStatus=status, sideEffects=effects)

    created, previous = [], None
    for index, step in enumerate(plan):
        try:
            kwargs = {'user_auth': True}
            if step['text'].strip(): kwargs['text'] = step['text']
            if media_ids[index]: kwargs['media_ids'] = media_ids[index]
            if previous: kwargs['in_reply_to_tweet_id'] = previous
            post_id = _x_post_id(client.create_tweet(**kwargs))
            created.append({'step': index + 1, 'clientId': step['clientId'], 'id': post_id,
                'url': f'https://x.com/i/web/status/{post_id}', 'replyToId': previous,
                'mediaIds': media_ids[index]})
            effects.append({'action': 'create_post', 'id': post_id, 'step': index + 1})
            previous = post_id
        except Exception as error:
            status, message = _x_error(error); unknown = status is None
            return _failure(message, 502, partial=bool(created), xAttempted=True, postAttempted=True,
                mediaUploadAttempted=bool(any(media)), safeToRetry=not created and not unknown,
                resultUnknown=unknown, failedStep=index + 1, publishedCount=len(created),
                xStatus=status, posts=created, sideEffects=effects)

    return {'live': True, 'success': True, 'complete': True, 'xAttempted': True, 'resultUnknown': False,
        'postAttempted': True, 'mediaUploadAttempted': bool(any(media)), 'safeToRetry': False,
        'postCount': len(created), 'publishedCount': len(created), 'rootPostId': created[0]['id'],
        'lastPostId': created[-1]['id'], 'posts': created, 'sideEffects': effects, 'retryPerformed': False}, 200

## 4.0 Live Twitter publishing route and browser configuration

In [ ]:
def install_social_publish(rt, public_domain=None, path=PUBLISH_ROUTE, app=None):
    """Install the capability-protected thread publishing route."""
    app = app or getattr(rt, '__self__', None)
    _drop_route(app, path)
    from starlette.responses import JSONResponse
    @rt(path)
    async def post(req):
        headers = _cors(req)
        def reject(message, status, **details):
            return JSONResponse({'live': True, 'success': False, 'error': message} | details,
                                status_code=status, headers=headers)
        uploads = {}
        try:
            if req.headers.get('content-type', '').lower().startswith('multipart/form-data'):
                try: form = await req.form(max_files=400, max_fields=2, max_part_size=MAX_LIVE_MEDIA + 1024)
                except TypeError: form = await req.form(max_files=400, max_fields=2)
                body = json.loads(str(form.get('payload') or ''))
                for key, value in form.multi_items():
                    if key.startswith('media_'):
                        if key in uploads: raise ValueError('Duplicate media upload field.')
                        uploads[key] = value
            else: body = json.loads((await req.body()).decode())
        except Exception:
            return reject('A JSON object is required.', 400)
        if not isinstance(body, dict):
            return reject('A JSON object is required.', 400)
        supplied = str(body.pop('publishToken', ''))
        if not supplied or not secrets.compare_digest(supplied, SOLVEIT_SOCIAL_PUBLISH_TOKEN):
            return reject('Publishing is not authorized.', 403)
        request_id = str(body.pop('requestId', ''))
        try: uuid.UUID(request_id)
        except (ValueError, AttributeError):
            return reject('A valid request ID is required.', 400)
        try:
            plan = _live_plan(body, len(uploads))
            expected = {f'media_{i}_{j}': (i, step_media) for i, step in enumerate(plan)
                        for j, step_media in enumerate(step['media'])}
            if set(uploads) != set(expected): raise ValueError('The uploaded media fields do not match the posting plan.')
            media = [[] for _ in plan]
            for key, (post_index, planned) in expected.items():
                media[post_index].append(await _read_live_media(uploads[key], planned))
        except (ValueError, TypeError) as error:
            return reject(str(error), 422, xAttempted=False, safeToRetry=True, resultUnknown=False,
                          posts=[], sideEffects=[], retryPerformed=False)
        if SOLVEIT_SOCIAL_PUBLISH_LOCK.locked():
            return reject('Another publish request is already running.', 409, xAttempted=False,
                          posts=[], sideEffects=[], retryPerformed=False)
        async with SOLVEIT_SOCIAL_PUBLISH_LOCK:
            if request_id in SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS:
                return reject('This publish request was already attempted.', 409, xAttempted=False,
                              posts=[], sideEffects=[], retryPerformed=False)
            SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS[request_id] = True
            while len(SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS) > 100: SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS.pop(next(iter(SOLVEIT_SOCIAL_PUBLISH_ATTEMPTS)))
            result, status = await asyncio.to_thread(_publish_plan, plan, None, None, media)
        return JSONResponse(result, status_code=status, headers=headers)
    iife(f'window.SOLVEIT_SOCIAL_PUBLISH_URL={json.dumps(_endpoint(public_domain, path))};'
         f'window.SOLVEIT_SOCIAL_PUBLISH_TOKEN={json.dumps(SOLVEIT_SOCIAL_PUBLISH_TOKEN)}')

install_social_publish(rt, public_domain, app=app)